# Qdrant Demo

### Importing Libraries

This notebook demonstrates how we will use Qdrant to store and retrieve embeddings. We begin by importing the neccesary libraires

In [26]:
from qdrant_client import QdrantClient
from qdrant_client.http.models import PointStruct, VectorParams, Distance
from ollama import Client as OllamaClient, ChatResponse
import requests
from typing import Mapping, Any, Optional, Sequence
import os

### Client Setup
Next we define functions to get Ollama and Qdrant clients

In [27]:
_ollama_client: Optional["OllamaClient"] = None

def _get_ollama_client() -> OllamaClient:
    global _ollama_client
    if _ollama_client is None:
        host = os.getenv("OLLAMA_HOST", "http://localhost:11434")
        _ollama_client = OllamaClient(host=host)
    return _ollama_client

In [28]:
_qdrant_client: Optional["QdrantClient"] = None

def _get_qdrant_client() -> QdrantClient:
    global _qdrant_client
    if _qdrant_client is None:
        url = os.getenv("QDRANT_HOST", "http://localhost:6333")
        _qdrant_client = QdrantClient(url=url)
    return _qdrant_client

### Embedding Function

This function accepts a strings and returns the corresponding embedding (vector) from nomic-text-embed running in Ollama

In [29]:
def embed(text:str) -> Sequence[float]:
    _ollama_client = _get_ollama_client()
    response: ChatResponse = _ollama_client.embed(model="nomic-embed-text", input=text) # type: ignore
    return response.embeddings[0] # type: ignore

### Qdrant Setup and Storing Embeddings

In [30]:
qdrant_client = _get_qdrant_client()

Create a collection of embeddings of appropriatte size in qdrant

In [31]:
qdrant_client.recreate_collection(
    collection_name="test_collection",
    vectors_config=VectorParams(size=768, distance=Distance.COSINE)
)

/var/folders/nb/0qk42j7j0bq93_44dknp53s00000gn/T/ipykernel_9677/3554724465.py:1: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  qdrant_client.recreate_collection(


True

Creating some sample documents:

In [32]:
documents = [
    "The capital of France is Paris.",
    "Machine learning enables computers to learn from data.",
    "Special Forces operations require precise intelligence."
]

Now embed our three sample documents and store them as Qdrant PointStruc objects

In [33]:
points = []
for i, doc in enumerate(documents):
    vec = embed(doc)
    points.append(
        PointStruct(
            id=i,
            vector=vec,
            payload={"text": doc}
        )
    )

Now we "upsert" our points to Qdrant. "upsert" = update + insert

In [34]:
qdrant_client.upsert(collection_name="test_collection", points=points)

UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

### Query Qdrant

Now we add a "query" string:

In [ ]:
user_query = embed("Tell me about France")

Pass the vector to qdrant and return the two closest documents

In [ ]:
results = qdrant_client.query_points(
    collection_name="test_collection",
    query=user_query,
    limit=1
)

Show the results:

In [42]:
print(results)
print(type(results))
print(results.points[0].payload["text"])

points=[ScoredPoint(id=0, version=1, score=0.6801282, payload={'text': 'The capital of France is Paris.'}, vector=None, shard_key=None, order_value=None)]
<class 'qdrant_client.http.models.models.QueryResponse'>
The capital of France is Paris.
